# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Which content items should an editor review first for refresh, expansion, or protection —
out of thousands — and does a model beat a transparent hand-written rule at ranking that
queue?**

Decision supported: where limited weekly editor attention goes first (Lane 2: Refresh / Content
Opportunity Scoring). Full reasoning in `w01_research_question.ipynb`.

## 2. Data

FlyRank ML Internship starter release: 30,000 content items across 32 clients, pseudonymous
IDs throughout. `trend_direction`/`trend_pct` excluded from every feature set (label-derived).
Full contract in `w03_data_contract.ipynb`.

In [1]:
import pandas as pd, numpy as np, json

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
print(f"{len(df):,} rows, {df['client_id'].nunique()} clients, base rate {df['is_declining_label'].mean():.3f}")

30,000 rows, 32 clients, base rate 0.542


## 3. Methodology

Baseline: transparent CTR-vs-position rule (staleness signal checked and found MIXED, so the
rule leans on the CONFIRMED CTR-vs-position signal instead — `w04_baseline_score.ipynb`).
Model: Logistic Regression + Random Forest on 13 pre-decision features, client-grouped holdout
split (6 of 32 clients held out) — `w05_model.ipynb`, re-audited in `w06_validation_audit.ipynb`.

In [2]:
with open('../outputs/action_playbook_metrics.json') as f:
    playbook_metrics = json.load(f)

print("Validated precision@50:", playbook_metrics['validated_precision_at_50'],
      "(source:", playbook_metrics['validated_precision_at_50_source'], ")")
print("Base rate:", playbook_metrics['base_rate'])

Validated precision@50: 0.7 (source: w06_validation_audit.ipynb, client-grouped holdout, 6 test clients )
Base rate: 0.542


## 4. Results (vs baseline)

| Method | precision@50 | precision@20% |
|---|---|---|
| Baseline (rule) | 0.460 | 0.555 |
| Logistic Regression | 0.220 | 0.559 |
| Random Forest | 0.700 | 0.609 |

Test base rate = 0.391, n(test) = 2,325 across 6 held-out clients. Naive random-split precision@50
was 0.900 — the 0.200 gap versus the honest 0.700 is itself a finding (`w06_validation_audit.ipynb`).
Leakage test: adding the label-derived `trend_pct` back pushed AUC from 0.741 to a perfect 1.000,
confirming the honest number is real.

In [3]:
results = pd.DataFrame({
    'method': ['Baseline (rule)', 'Logistic Regression', 'Random Forest'],
    'precision@50': [0.460, 0.220, 0.700],
    'precision@20%': [0.555, 0.559, 0.609],
})
results

,method,precision@50,precision@20%
0,Baseline (rule),0.46,0.555
1,Logistic Regression,0.22,0.559
2,Random Forest,0.70,0.609


## 5. Limitations

- The label is a proxy (30-day impression trend), not a confirmed editorial judgment.
- Only 6 held-out clients back the validated precision numbers — directional, not conclusive.
- This dataset's decay pattern (decline rate *falls* with content age) runs opposite to
  FlyRank's own published research paper's finding — stated honestly, not resolved.
- No causal claims anywhere — everything here is observed, measured, or decision-support
  language.

## 6. Ranked recommendations

Four archetypes (k-means, k=4), each mapped to a distinct action — not one rule for everyone.
Full playbook, human-review rules, and no-go list in `w07_action_playbook.ipynb`.

In [4]:
for row in playbook_metrics['archetype_profile']:
    name = playbook_metrics['archetype_names'][str(row['archetype_id'])]
    action = playbook_metrics['archetype_actions'][str(row['archetype_id'])]
    print(f"{name:28s} n={row['n']:<7,} decline_rate={row['decline_rate']:.2f}  -> {action}")

Visible but Thin             n=9,442   decline_rate=0.52  -> expand_and_refresh
High-Visibility Decliners    n=12,205  decline_rate=0.63  -> refresh_priority
Low-Volume Outliers          n=159     decline_rate=0.16  -> exclude_low_confidence
Fresh but Low-Reach          n=8,194   decline_rate=0.44  -> monitor_promote


## 7. Artifacts the paper embeds

Figures and metrics referenced by the deployed paper (`docs/index.html`), generated across
Weeks 4-7 and committed to `work/figures/` and `work/outputs/`.

In [5]:
import os
print("Figures:", os.listdir('../figures'))
print("Committed metrics:", [f for f in os.listdir('../outputs') if f.endswith('.json')])

Figures: ['w07_action_playbook_overview.png', 'decay_by_age.png']
Committed metrics: ['playbook_metrics.json', 'action_playbook_metrics.json']


## 8. Five-minute demo outline (Week-8 showcase, optional)

**Question (30s):** Out of 30,000 content pages across 32 client accounts, which ones should an
editor review first this week — and does a model actually beat a simple hand-written rule at
ranking that queue, or does it just look like it does?

**Method (60s):** Built a transparent CTR-vs-position rule as the baseline, after checking two
candidate signals against real data first (staleness came back MIXED, CTR-vs-position came back
CONFIRMED). Trained Logistic Regression and Random Forest against it, using a client-grouped
holdout — 6 of 32 clients held out entirely, so the test never sees a client the model trained
on.

**One chart (90s):** the model-vs-baseline bar chart (`docs/assets/model_vs_baseline.png`).
Point at three things: Random Forest's 0.70 vs. the rule's 0.46 at precision@50; Logistic
Regression actually losing to the rule at that same cut; and the dashed base-rate line, so the
room can see none of these numbers are being flattered by a lucky class balance.

**One honest result (90s):** the leakage jump — 0.741 to a perfect 1.000 AUC the moment the
label-derived column goes back in as a feature (`docs/assets/leakage_jump.png`). This is the
slide that proves the honest number is real: if I can make the model look perfect on command
just by cheating, and I *didn't* do that, the room can trust the 0.70.

**One recommendation (30s):** four archetypes, four different actions — not "flag everything,"
but a `refresh_priority` queue for the 12,205 high-visibility decliners specifically, with a
human-review checklist before anyone acts on it, and a named no-go list for what should never
be automated.

## 9. Two shareable cuts

**Social post (methodology-focused):**

> Spent 8 weeks building a content-refresh scoring model for a real SEO dataset — 30,000 pages,
> 32 clients. The most useful result wasn't the model beating my baseline (it did, 0.70 vs 0.46
> precision@50). It was catching my own mistake first: running the same model on a random
> train/test split instead of a client-grouped one inflated the score to 0.90 — a full 0.20
> points of pure memorization, not skill. The honest number is the one that survived a
> deliberately harder test. Full writeup + reproducible notebooks: [paper URL]

**Employer-facing summary (3 sentences):**

> I built a content-refresh priority model for a real 30,000-page SEO dataset spanning 32
> client accounts, comparing Logistic Regression and Random Forest against a transparent
> hand-written baseline under a client-grouped validation split. The model beat the baseline at
> ranking the highest-priority pages (precision@50 of 0.70 vs. 0.46), while a deliberate leakage
> test confirmed the result was genuine rather than inflated by a naive split or a
> label-derived feature. The output is a ranked, human-reviewed action playbook with named
> limitations — not an automated system — reproducible end-to-end from a public GitHub repo.